In [ ]:
# ETAPA 3 - PRÉ-PROCESSAMENTO DE DADOS

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

caminho_arquivo = "dados-empresa.csv"

df_original = pd.read_csv(caminho_arquivo)

df_tratado = df_original.copy()

df_original.shape

(1020, 27)

In [ ]:
# FUNÇÃO DE DIAGNÓSTICO DA QUALIDADE DOS DADOS

def diagnostico_qualidade(base):
    
    # Converter temporariamente a coluna data para verificar datas inválidas
    datas_convertidas = pd.to_datetime(base["data"], errors="coerce")
    
    # Verificar inconsistência na receita
    receita_calculada = base["quantidade"] * base["preco_unitario"] * (1 - base["desconto"])
    inconsistencias_receita = ((base["receita"] - receita_calculada).abs() > 0.01).sum()
    
    # Verificar inconsistência no lucro
    lucro_calculado = base["receita"] - (base["quantidade"] * base["custo_unitario"])
    inconsistencias_lucro = ((base["lucro"] - lucro_calculado).abs() > 0.01).sum()
    
    # Verificar inconsistência no atingimento de meta
    atingiu_meta_calculado = np.where(base["receita"] >= base["meta_venda"], "Sim", "Não")
    inconsistencias_meta = (base["atingiu_meta"] != atingiu_meta_calculado).sum()
    
    resumo = {
        "linhas": base.shape[0],
        "colunas": base.shape[1],
        "linhas_duplicadas": base.duplicated().sum(),
        "ids_venda_duplicados": base["id_venda"].duplicated().sum(),
        "total_valores_nulos": base.isna().sum().sum(),
        "datas_invalidas": datas_convertidas.isna().sum(),
        "quantidade_menor_ou_igual_zero": (base["quantidade"] <= 0).sum(),
        "registros_regiao_exterior": (base["regiao"] == "Exterior").sum(),
        "inconsistencias_receita": inconsistencias_receita,
        "inconsistencias_lucro": inconsistencias_lucro,
        "inconsistencias_meta": inconsistencias_meta
    }
    
    return pd.DataFrame(resumo.items(), columns=["problema", "quantidade"])

In [ ]:
diagnostico_antes = diagnostico_qualidade(df_original)

diagnostico_antes

,problema,quantidade
0,linhas,1020
1,colunas,27
2,linhas_duplicadas,20
3,ids_venda_duplicados,20
4,total_valores_nulos,204
5,datas_invalidas,0
6,quantidade_menor_ou_igual_zero,10
7,registros_regiao_exterior,30
8,inconsistencias_receita,10
9,inconsistencias_lucro,10


In [4]:
# TRATAMENTO 1 - REMOVER LINHAS DUPLICADAS

linhas_antes = df_tratado.shape[0]

df_tratado = df_tratado.drop_duplicates().reset_index(drop=True)

linhas_depois = df_tratado.shape[0]

print(f"Linhas antes da remoção de duplicidades: {linhas_antes}")
print(f"Linhas depois da remoção de duplicidades: {linhas_depois}")
print(f"Linhas removidas: {linhas_antes - linhas_depois}")

Linhas antes da remoção de duplicidades: 1020
Linhas depois da remoção de duplicidades: 1000
Linhas removidas: 20


In [ ]:
# TRATAMENTO 2 - CONVERTER A COLUNA DATA PARA DATETIME

df_tratado["data"] = pd.to_datetime(df_tratado["data"], errors="coerce")

# Verificar se houve datas inválidas após a conversão
datas_invalidas = df_tratado["data"].isna().sum()

print(f"Datas inválidas após conversão: {datas_invalidas}")
print(df_tratado["data"].dtype)

Datas inválidas após conversão: 0
datetime64[us]


In [6]:
# TRATAMENTO 3 - CORRIGIR REGIÃO COM BASE NA CIDADE

# Mapa correto de cidade para região brasileira
mapa_cidade_regiao = {
    "Manaus": "Norte",
    "Belém": "Norte",
    "Salvador": "Nordeste",
    "Recife": "Nordeste",
    "São Paulo": "Sudeste",
    "Rio de Janeiro": "Sudeste",
    "Curitiba": "Sul",
    "Porto Alegre": "Sul",
    "Brasília": "Centro-Oeste",
    "Goiânia": "Centro-Oeste"
}

# Quantidade de registros com região Exterior antes
regiao_exterior_antes = (df_tratado["regiao"] == "Exterior").sum()

# Corrigir a região com base na cidade
df_tratado["regiao"] = df_tratado["cidade"].map(mapa_cidade_regiao).fillna(df_tratado["regiao"])

# Quantidade de registros com região Exterior depois
regiao_exterior_depois = (df_tratado["regiao"] == "Exterior").sum()

print(f"Registros com região 'Exterior' antes: {regiao_exterior_antes}")
print(f"Registros com região 'Exterior' depois: {regiao_exterior_depois}")

Registros com região 'Exterior' antes: 30
Registros com região 'Exterior' depois: 0


In [7]:
# TRATAMENTO 4 - CORRIGIR QUANTIDADE IGUAL A ZERO

# Identificar registros com quantidade igual a zero e informações suficientes para correção
condicao_quantidade_zero = (
    (df_tratado["quantidade"] == 0) &
    (df_tratado["preco_unitario"].notna()) &
    (df_tratado["desconto"].notna()) &
    (df_tratado["preco_unitario"] > 0) &
    (df_tratado["desconto"] < 1) &
    (df_tratado["receita"].notna()) &
    (df_tratado["receita"] > 0)
)

qtd_zero_antes = (df_tratado["quantidade"] == 0).sum()

# Calcular a quantidade correta
quantidade_corrigida = (
    df_tratado.loc[condicao_quantidade_zero, "receita"] /
    (
        df_tratado.loc[condicao_quantidade_zero, "preco_unitario"] *
        (1 - df_tratado.loc[condicao_quantidade_zero, "desconto"])
    )
)

# Aplicar a correção arredondando para inteiro
df_tratado.loc[condicao_quantidade_zero, "quantidade"] = quantidade_corrigida.round().astype(int)

qtd_zero_depois = (df_tratado["quantidade"] == 0).sum()

print(f"Registros com quantidade igual a zero antes: {qtd_zero_antes}")
print(f"Registros com quantidade igual a zero depois: {qtd_zero_depois}")

Registros com quantidade igual a zero antes: 10
Registros com quantidade igual a zero depois: 0


In [8]:
# TRATAMENTO 5 - PREENCHER VALORES NULOS EM DESCONTO

def preencher_numerico_por_mediana_grupo(base, coluna, grupos):
    """
    Preenche valores nulos de uma coluna numérica usando a mediana
    de grupos semelhantes. Caso ainda sobrem nulos, usa a mediana global.
    """
    
    base = base.copy()
    
    for grupo in grupos:
        base[coluna] = base[coluna].fillna(
            base.groupby(grupo)[coluna].transform("median")
        )
    
    # Caso ainda exista algum nulo, preencher com a mediana global
    base[coluna] = base[coluna].fillna(base[coluna].median())
    
    return base[coluna]


nulos_desconto_antes = df_tratado["desconto"].isna().sum()

df_tratado["desconto"] = preencher_numerico_por_mediana_grupo(
    base=df_tratado,
    coluna="desconto",
    grupos=[
        ["produto", "categoria", "linha_produto"],
        ["produto"],
        ["categoria"]
    ]
)

nulos_desconto_depois = df_tratado["desconto"].isna().sum()

print(f"Nulos em desconto antes: {nulos_desconto_antes}")
print(f"Nulos em desconto depois: {nulos_desconto_depois}")

Nulos em desconto antes: 50
Nulos em desconto depois: 0


In [9]:
# TRATAMENTO 6 - RECALCULAR PREÇO UNITÁRIO AUSENTE

nulos_preco_antes = df_tratado["preco_unitario"].isna().sum()

condicao_preco_nulo = df_tratado["preco_unitario"].isna()

df_tratado.loc[condicao_preco_nulo, "preco_unitario"] = (
    df_tratado.loc[condicao_preco_nulo, "receita"] /
    (
        df_tratado.loc[condicao_preco_nulo, "quantidade"] *
        (1 - df_tratado.loc[condicao_preco_nulo, "desconto"])
    )
)

nulos_preco_depois = df_tratado["preco_unitario"].isna().sum()

print(f"Nulos em preço unitário antes: {nulos_preco_antes}")
print(f"Nulos em preço unitário depois: {nulos_preco_depois}")

Nulos em preço unitário antes: 50
Nulos em preço unitário depois: 0


In [ ]:
# TRATAMENTO 7 - RECALCULAR CUSTO UNITÁRIO AUSENTE

nulos_custo_antes = df_tratado["custo_unitario"].isna().sum()

condicao_custo_nulo = df_tratado["custo_unitario"].isna()

df_tratado.loc[condicao_custo_nulo, "custo_unitario"] = (
    (
        df_tratado.loc[condicao_custo_nulo, "receita"] -
        df_tratado.loc[condicao_custo_nulo, "lucro"]
    ) /
    df_tratado.loc[condicao_custo_nulo, "quantidade"]
)

nulos_custo_depois = df_tratado["custo_unitario"].isna().sum()

print(f"Nulos em custo unitário antes: {nulos_custo_antes}")
print(f"Nulos em custo unitário depois: {nulos_custo_depois}")

Nulos em custo unitário antes: 50
Nulos em custo unitário depois: 0


In [11]:
# TRATAMENTO 8 - PREENCHER FAIXA DE RENDA AUSENTE

def preencher_categorico_por_moda_grupo(base, coluna, grupos):
    """
    Preenche valores nulos de uma coluna categórica usando a moda
    de grupos semelhantes. Caso ainda sobrem nulos, usa a moda global.
    """
    
    base = base.copy()
    
    for grupo in grupos:
        moda_por_grupo = base.groupby(grupo)[coluna].transform(
            lambda serie: serie.mode(dropna=True).iloc[0]
            if not serie.mode(dropna=True).empty
            else np.nan
        )
        
        base[coluna] = base[coluna].fillna(moda_por_grupo)
    
    # Caso ainda exista algum nulo, preencher com a moda global
    moda_global = base[coluna].mode(dropna=True).iloc[0]
    base[coluna] = base[coluna].fillna(moda_global)
    
    return base[coluna]


nulos_faixa_antes = df_tratado["faixa_renda"].isna().sum()

df_tratado["faixa_renda"] = preencher_categorico_por_moda_grupo(
    base=df_tratado,
    coluna="faixa_renda",
    grupos=[
        ["tipo_cliente", "segmento"],
        ["tipo_cliente"],
        ["segmento"]
    ]
)

nulos_faixa_depois = df_tratado["faixa_renda"].isna().sum()

print(f"Nulos em faixa_renda antes: {nulos_faixa_antes}")
print(f"Nulos em faixa_renda depois: {nulos_faixa_depois}")

Nulos em faixa_renda antes: 50
Nulos em faixa_renda depois: 0


In [ ]:
# TRATAMENTO 9 - RECALCULAR CAMPOS DERIVADOS

# Recalcular mês e trimestre a partir da data
df_tratado["mes"] = df_tratado["data"].dt.month
df_tratado["trimestre"] = df_tratado["data"].dt.quarter

# Recalcular atingimento de meta
df_tratado["atingiu_meta"] = np.where(
    df_tratado["receita"] >= df_tratado["meta_venda"],
    "Sim",
    "Não"
)

# Verificação rápida
df_tratado[["data", "mes", "trimestre", "receita", "meta_venda", "atingiu_meta"]].head()

,data,mes,trimestre,receita,meta_venda,atingiu_meta
0,2024-12-14,12,4,"19,258.19","24,931.11",Não
1,2024-03-09,3,1,"4,457.55","5,780.24",Não
2,2024-02-14,2,1,362.16,554.46,Não
3,2024-07-20,7,3,"21,446.22","30,872.49",Não
4,2024-07-11,7,3,"21,105.90","24,448.64",Não


In [ ]:
# TRATAMENTO 10 - PADRONIZAR TIPOS DE DADOS

# Garantir que colunas inteiras estejam como inteiros
colunas_inteiras = [
    "id_venda",
    "mes",
    "trimestre",
    "tempo_relacionamento_meses",
    "quantidade",
    "prazo_entrega_dias"
]

for coluna in colunas_inteiras:
    df_tratado[coluna] = df_tratado[coluna].astype(int)

# Garantir que colunas financeiras estejam como float
colunas_float = [
    "preco_unitario",
    "desconto",
    "receita",
    "custo_unitario",
    "lucro",
    "meta_venda"
]

for coluna in colunas_float:
    df_tratado[coluna] = df_tratado[coluna].astype(float)

# Verificar tipos finais
df_tratado.dtypes

id_venda                               int64
data                          datetime64[us]
mes                                    int64
trimestre                              int64
cliente                                  str
tipo_cliente                             str
tempo_relacionamento_meses             int64
faixa_renda                              str
segmento                                 str
produto                                  str
categoria                                str
marca                                    str
linha_produto                            str
quantidade                             int64
preco_unitario                       float64
desconto                             float64
receita                              float64
custo_unitario                       float64
lucro                                float64
regiao                                   str
cidade                                   str
canal_venda                              str
vendedor  

In [ ]:
# EVIDÊNCIA FINAL - DIAGNÓSTICO APÓS O TRATAMENTO

diagnostico_depois = diagnostico_qualidade(df_tratado)

diagnostico_depois

,problema,quantidade
0,linhas,1000
1,colunas,27
2,linhas_duplicadas,0
3,ids_venda_duplicados,0
4,total_valores_nulos,0
5,datas_invalidas,0
6,quantidade_menor_ou_igual_zero,0
7,registros_regiao_exterior,0
8,inconsistencias_receita,0
9,inconsistencias_lucro,0


In [ ]:
# COMPARAÇÃO ENTRE BASE ORIGINAL E BASE TRATADA

comparacao_qualidade = diagnostico_antes.merge(
    diagnostico_depois,
    on="problema",
    suffixes=("_antes", "_depois")
)

comparacao_qualidade

,problema,quantidade_antes,quantidade_depois
0,linhas,1020,1000
1,colunas,27,27
2,linhas_duplicadas,20,0
3,ids_venda_duplicados,20,0
4,total_valores_nulos,204,0
5,datas_invalidas,0,0
6,quantidade_menor_ou_igual_zero,10,0
7,registros_regiao_exterior,30,0
8,inconsistencias_receita,10,0
9,inconsistencias_lucro,10,0


In [ ]:
# VERIFICAÇÃO FINAL DAS FÓRMULAS FINANCEIRAS

# Receita esperada
receita_calculada = (
    df_tratado["quantidade"] *
    df_tratado["preco_unitario"] *
    (1 - df_tratado["desconto"])
)

# Lucro esperado
lucro_calculado = (
    df_tratado["receita"] -
    (df_tratado["quantidade"] * df_tratado["custo_unitario"])
)

# Diferenças
df_tratado["diferenca_receita"] = (df_tratado["receita"] - receita_calculada).abs()
df_tratado["diferenca_lucro"] = (df_tratado["lucro"] - lucro_calculado).abs()

print("Inconsistências finais na receita:", (df_tratado["diferenca_receita"] > 0.01).sum())
print("Inconsistências finais no lucro:", (df_tratado["diferenca_lucro"] > 0.01).sum())

Inconsistências finais na receita: 0
Inconsistências finais no lucro: 0


In [18]:
# Remover colunas auxiliares criadas apenas para validação
df_tratado = df_tratado.drop(columns=["diferenca_receita", "diferenca_lucro"])

In [ ]:
# REDUÇÃO DE DADOS

print(f"Quantidade final de colunas: {df_tratado.shape[1]}")
print("Nenhuma coluna foi removida, pois todas podem contribuir para as análises de BI.")

Quantidade final de colunas: 27
Nenhuma coluna foi removida, pois todas podem contribuir para as análises de BI.


In [ ]:
# EXPORTAR BASE TRATADA

caminho_saida = "dados_empresa_tratado.csv"

df_tratado.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"Base tratada exportada com sucesso: {caminho_saida}")

Base tratada exportada com sucesso: dados_empresa_tratado.csv


# --- Etapa 3 - Pre-processamento ---

Nesta etapa foi realizado o pré-processamento da base de dados com o objetivo de corrigir os problemas identificados na fase de diagnóstico e preparar os dados para análises confiáveis de Business Intelligence.

A base original continha 1020 registros e 27 atributos. Durante a análise de qualidade dos dados, foram identificados problemas relevantes, como registros duplicados, valores ausentes, inconsistências regionais, registros com quantidade igual a zero e divergências nas fórmulas financeiras de receita e lucro. Esses problemas poderiam comprometer diretamente os indicadores de BI, como receita total, lucro total, margem de lucro, ticket médio, desempenho por região, desempenho por vendedor e análise de canais de venda.

Por esse motivo, as decisões de tratamento foram tomadas devido as seguintes justificativas:


### Remoção de registros duplicados

Foram identificadas 20 linhas completamente duplicadas na base original. A remoção desses registros foi necessária porque duplicidades alteram diretamente os indicadores agregados utilizados em Business Intelligence.

Como muitos KPIs são calculados por somatório, por exemplo:
    Receita Total = soma das receitas de todas as vendas
    Lucro Total = soma dos lucros de todas as vendas
    Quantidade Vendida = soma das quantidades vendidas

A presença de registros duplicados faz com que uma mesma venda seja contabilizada mais de uma vez. Isso gera uma superestimação artificial dos resultados. Esse erro impacta não apenas os totais, mas também médias e proporções, pois registros duplicados aumentam indevidamente o peso de determinadas observações dentro da amostra. 

Portanto, a decisão de remover apenas duplicidades completas foi considerada estatisticamente adequada, pois elimina repetições comprovadas sem descartar registros únicos e potencialmente válidos.

Após esse tratamento, a base foi reduzida de 1020 para 1000 registros.


### Conversão da coluna de data

A coluna data foi originalmente interpretada pelo Pandas como texto, ou seja, como tipo str. No entanto, conceitualmente ela representa uma variável temporal.

A conversão para o tipo datetime foi necessária porque análises temporais dependem de operações específicas sobre datas, como extração de mês, trimestre, ordenação cronológica e construção de séries temporais. Sem essa conversão, a data seria tratada apenas como uma sequência de caracteres, o que poderia gerar interpretações incorretas em análises como:
    Receita por mês
    Lucro por trimestre
    Evolução temporal das vendas
    Sazonalidade de desempenho

Após a conversão, foi possível validar que não existiam datas inválidas na base. Além disso, os campos mes e trimestre puderam ser recalculados diretamente a partir da coluna data, garantindo consistência temporal.


### Correção da região com base na cidade

Durante o diagnóstico, foram encontrados 30 registros classificados com a região 'Exterior'. No entanto, ao analisar a coluna cidade, verificou-se que esses registros estavam associados a cidades brasileiras, como São Paulo, Rio de Janeiro, Recife, Salvador, Manaus, Belém, Curitiba, Brasília e Goiânia.

Essa situação indica uma inconsistência categórica na variável regiao, pois o valor Exterior não é compatível com cidades localizadas no Brasil. A decisão adotada foi corrigir a região com base na cidade, utilizando um mapeamento direto entre cidades brasileiras e suas respectivas regiões geográficas.  Essa decisão é justificável porque a cidade é uma informação mais específica do que a região. Em termos de hierarquia geográfica:
    Cidade → Região

ou seja, sabendo a cidade, é possível determinar de forma objetiva a região correspondente. Portanto, essa correção não foi uma imputação estatística incerta, mas sim uma correção lógica baseada em uma relação geográfica conhecida. Por exemplo:
    São Paulo → Sudeste
    Recife → Nordeste
    Manaus → Norte
    Curitiba → Sul
    Goiânia → Centro-Oeste

A manutenção da categoria Exterior prejudicaria análises regionais, pois criaria uma região artificial e distorceria os resultados de receita, lucro e margem por região. Após o tratamento, todos os registros passaram a estar associados a regiões brasileiras válidas.


### Correção dos registros com quantidade igual a zero

Foram identificados 10 registros com quantidade igual a zero, mas com valores positivos de receita e lucro. Essa é uma inconsistência lógica, pois uma venda com quantidade zero não deveria gerar receita.

A princípio, esses registros poderiam ser removidos. No entanto, como eles possuíam informações financeiras suficientes, optou-se por recuperar a quantidade real vendida a partir da fórmula da receita.

A receita pode ser representada por:
    Receita = Quantidade × Preço Unitário × (1 - Desconto)

Reorganizando a fórmula, a quantidade pode ser calculada por:
    Quantidade = Receita / [Preço Unitário × (1 - Desconto)]

Essa decisão foi matematicamente mais adequada do que excluir os registros, pois permitiu recuperar a variável inconsistente usando uma relação determinística entre os próprios atributos da venda. A exclusão desses registros reduziria a amostra e eliminaria vendas que, apesar do erro na quantidade, ainda continham informações úteis sobre cliente, produto, receita, lucro, vendedor, região e canal de venda. Além disso, como a quantidade é uma variável discreta, o valor calculado foi arredondado para inteiro. Após essa correção, não restaram registros com quantidade igual a zero e as inconsistências nas fórmulas de receita e lucro foram eliminadas.


### Tratamento dos valores ausentes na coluna desconto

A coluna desconto possuía valores ausentes. Como o desconto é uma variável numérica percentual, seria inadequado preencher os valores faltantes de forma arbitrária.

A estratégia adotada foi preencher os valores ausentes utilizando a mediana de grupos semelhantes, considerando inicialmente combinações de produto, categoria e linha_produto. A mediana foi escolhida em vez da média porque é uma medida estatística mais robusta contra valores extremos. Enquanto a média pode ser fortemente influenciada por descontos muito altos ou muito baixos, a mediana representa o valor central da distribuição. Do ponto de vista estatístico, a média minimiza os erros quadráticos, enquanto a mediana minimiza os desvios absolutos. Em bases comerciais, onde podem existir promoções específicas, descontos agressivos ou condições atípicas, a mediana tende a representar melhor o comportamento típico dos dados. Além disso, o preenchimento foi feito por grupos semelhantes porque descontos costumam variar de acordo com características comerciais do produto. Um notebook premium, por exemplo, pode ter uma política de desconto diferente de um mouse econômico.

A ordem de preenchimento foi:
    1. Mediana por produto, categoria e linha de produto
    2. Mediana por produto
    3. Mediana por categoria
    4. Mediana global

Essa abordagem reduz o risco de distorção, pois prioriza primeiro grupos mais específicos e só utiliza medidas mais gerais quando não há informação suficiente no grupo.


### Recalcular valores ausentes de preco_unitario

A coluna preco_unitario também possuía valores ausentes. Nesse caso, optou-se por recuperar matematicamente a partir da fórmula da receita. A fórmula original é:
    Receita = Quantidade × Preço Unitário × (1 - Desconto)

Isolando o preço unitário:
    Preço Unitário = Receita / [Quantidade × (1 - Desconto)]

Essa decisão é mais precisa do que uma imputação estatística porque utiliza informações reais da própria venda. Ou seja, em vez de estimar o preço com base em outras vendas, o valor foi reconstruído a partir da relação matemática entre receita, quantidade e desconto. Isso preserva a coerência financeira da base e evita distorções em análises futuras de preço médio, receita por produto, margem e lucratividade.


### Recalcular valores ausentes de custo_unitario

A coluna custo_unitario também apresentava valores ausentes. Assim como no caso do preço unitário, foi possível recuperar esse valor por meio de uma relação matemática.

O lucro pode ser representado por:
    Lucro = Receita - (Quantidade × Custo Unitário)

Isolando o custo unitário:
    Custo Unitário = (Receita - Lucro) / Quantidade

Essa abordagem foi considerada a melhor opção porque o custo unitário não foi estimado por média ou mediana, mas sim recalculado a partir dos valores financeiros já existentes na venda. Essa decisão mantém a consistência da seguinte relação:
    Receita - Custo Total = Lucro
onde:
    Custo Total = Quantidade × Custo Unitário

Dessa forma, a reconstrução do custo unitário preserva a lógica contábil da base e garante que os cálculos de margem e lucro permaneçam coerentes.


### Tratamento dos valores ausentes em faixa_renda

A variável faixa_renda é categórica ordinal, pois seus valores representam categorias com uma ordem natural, como baixa, média e alta. Como se trata de uma variável categórica, não seria adequado utilizar média ou mediana numérica. A estratégia mais apropriada foi o preenchimento pela moda, ou seja, pela categoria mais frequente. No entanto, em vez de utilizar diretamente a moda global da base, foi aplicada uma imputação por grupos semelhantes, considerando variáveis como tipo_cliente e segmento. Essa decisão reduz o risco de generalização excessiva. Por exemplo, clientes recorrentes de determinado segmento podem ter um perfil de renda diferente de clientes novos de outro segmento. Assim, preencher a faixa de renda considerando grupos semelhantes tende a preservar melhor a estrutura original dos dados.

A ordem utilizada foi:
    1. Moda por tipo de cliente e segmento
    2. Moda por tipo de cliente
    3. Moda por segmento
    4. Moda global

A moda foi escolhida porque representa a categoria mais provável dentro de um grupo. Estatisticamente, para variáveis qualitativas, a moda é a medida de tendência central mais adequada


### Recalcular campos derivados

Após os tratamentos principais, os campos mes, trimestre e atingiu_meta foram recalculados. Essa decisão foi tomada porque esses atributos são derivados de outras variáveis da base.

O mês e o trimestre derivam diretamente da data:
    Mês = mês extraído da data
    Trimestre = trimestre extraído da data

Já o campo atingiu_meta depende da comparação entre receita e meta_venda:
    Se Receita >= Meta de Venda → Atingiu Meta = Sim
    Se Receita < Meta de Venda → Atingiu Meta = Não

Mesmo que esses campos não apresentassem inconsistências graves inicialmente, recalculá-los após o tratamento aumenta a confiabilidade da base. Essa prática reduz o risco de manter variáveis derivadas desatualizadas após correções em campos financeiros ou temporais.


### Decisão sobre os outliers

Na Etapa 2 foram identificados possíveis outliers em variáveis como receita, lucro e meta_venda, utilizando o método do intervalo interquartil. No entanto, os outliers não foram removidos automaticamente. Essa decisão foi tomada porque, no contexto de vendas, valores extremos podem representar eventos reais de negócio. Uma venda com receita muito alta pode corresponder a uma compra corporativa ou a uma venda em maior volume. Um lucro negativo pode indicar uma venda com prejuízo, desconto elevado ou custo superior à receita. Uma meta muito alta pode representar uma meta diferenciada para determinado vendedor, região ou canal. Portanto, remover outliers apenas por critério estatístico poderia eliminar informações relevantes para a tomada de decisão. A decisão mais adequada foi manter esses registros e analisá-los posteriormente, principalmente na Etapa 4, quando serão feitos cruzamentos entre variáveis e análises de desempenho. Essa abordagem é importante porque outliers podem ser justamente os casos mais relevantes para a análise gerencial, especialmente quando indicam oportunidades, riscos ou problemas operacionais.


### Decisão sobre redução de dados

A redução de dados era opcional nesta etapa. A decisão foi não remover colunas da base. Essa escolha foi tomada porque todos os atributos possuem potencial analítico para as próximas etapas do projeto. Variáveis de cliente, produto, região, canal, vendedor, metas e entrega podem contribuir para diferentes perguntas de negócio. A remoção antecipada de colunas poderia limitar análises futuras, como:
    Produto × Região
    Canal × Lucro
    Vendedor × Meta
    Cliente × Receita
    Status de entrega × Desempenho comercial
    Faixa de renda × Tipo de produto

Portanto, optou-se por manter todas as colunas, preservando a capacidade analítica da base para as etapas de análise exploratória, modelagem dimensional e construção do dashboard.
